In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('/opt/workspace')\n",
    "\n",
    "from datetime import datetime\n",
    "from pyspark.sql import SparkSession\n",
    "from pyspark.sql.functions import (\n",
    "    col, lit, concat_ws, sha2, to_json, struct, current_timestamp\n",
    ")\n",
    "from config.settings import oracle_config, clickhouse_config, spark_config, app_config"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ref_date = datetime.now().strftime(\"%Y-%m-%d\")\n",
    "ref_date"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "spark = (\n",
    "    SparkSession.builder\n",
    "    .appName(f\"BronzeIngestion_{ref_date}\")\n",
    "    .config(\"spark.driver.memory\", spark_config.driver_memory)\n",
    "    .config(\"spark.executor.memory\", spark_config.executor_memory)\n",
    "    .config(\"spark.executor.cores\", spark_config.executor_cores)\n",
    "    .config(\"spark.sql.shuffle.partitions\", spark_config.sql_shuffle_partitions)\n",
    "    .config(\"spark.sql.adaptive.enabled\", spark_config.sql_adaptive_enabled)\n",
    "    .config(\"spark.sql.adaptive.coalescePartitions.enabled\", \"true\")\n",
    "    .config(\"spark.jars\", \"/opt/spark/jars/ojdbc11.jar,/opt/spark/jars/clickhouse-jdbc.jar\")\n",
    "    .getOrCreate()\n",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "oracle_jdbc_url = f\"jdbc:oracle:thin:@{oracle_config.host}:{oracle_config.port}/{oracle_config.service}\"\n",
    "oracle_jdbc_url"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "schema_name = \"ADMBI_PRD\"\n",
    "table_name = \"YOUR_TABLE_NAME\"\n",
    "primary_keys = [\"ID\"]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df_oracle = (\n",
    "    spark.read\n",
    "    .format(\"jdbc\")\n",
    "    .option(\"url\", oracle_jdbc_url)\n",
    "    .option(\"dbtable\", f\"{schema_name}.{table_name}\")\n",
    "    .option(\"user\", oracle_config.user)\n",
    "    .option(\"password\", oracle_config.password)\n",
    "    .option(\"driver\", \"oracle.jdbc.driver.OracleDriver\")\n",
    "    .option(\"fetchsize\", app_config.batch_size)\n",
    "    .option(\"numPartitions\", \"10\")\n",
    "    .load()\n",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df_oracle.printSchema()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "primary_key_str = concat_ws(\"|||\", *[col(pk) for pk in primary_keys])\n",
    "all_columns = [col(c) for c in df_oracle.columns]\n",
    "row_data = to_json(struct(*all_columns))\n",
    "row_hash_input = concat_ws(\"|||\", *all_columns)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df_bronze = df_oracle.select(\n",
    "    lit(ref_date).cast(\"date\").alias(\"ref_date\"),\n",
    "    lit(table_name).alias(\"table_name\"),\n",
    "    primary_key_str.alias(\"primary_key\"),\n",
    "    sha2(row_hash_input, 256).alias(\"row_hash\"),\n",
    "    row_data.alias(\"data\"),\n",
    "    current_timestamp().alias(\"ingestion_timestamp\")\n",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df_bronze.show(5, truncate=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "protocol = \"https\" if clickhouse_config.secure else \"http\"\n",
    "clickhouse_jdbc_url = (\n",
    "    f\"jdbc:clickhouse://{protocol}://{clickhouse_config.host}:\"\n",
    "    f\"{clickhouse_config.port}/{clickhouse_config.database}\"\n",
    ")\n",
    "clickhouse_jdbc_url"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df_bronze.write \\\n",
    "    .format(\"jdbc\") \\\n",
    "    .option(\"url\", clickhouse_jdbc_url) \\\n",
    "    .option(\"dbtable\", \"bronze.snapshot_raw\") \\\n",
    "    .option(\"user\", clickhouse_config.user) \\\n",
    "    .option(\"password\", clickhouse_config.password) \\\n",
    "    .option(\"driver\", \"com.clickhouse.jdbc.ClickHouseDriver\") \\\n",
    "    .option(\"batchsize\", app_config.batch_size) \\\n",
    "    .mode(\"append\") \\\n",
    "    .save()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "spark.stop()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.14"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}